# AgendaWatch Election-Keyword Analysis

**Purpose:** Surface and characterise election-related discussions in local government
meeting agendas/minutes from Colorado, Texas, and Missouri.

**Three analytical lenses:**
1. **Semantic neighbourhood** – what words live in the same vector space as 'election'?
2. **Temporal trends** – how has election-related language grown or declined over time?
3. **Geographic distribution** – which counties/cities mention elections most?

---

## 0. Setup

In [ ]:
# Run once to install dependencies
# !pip install -r ../requirements.txt

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook'  # or 'browser' to open plots in a tab

from main import ElectionAnalyzer
import config

KEYWORD = 'election'   # ← change this to explore other keywords
STATES  = ['CO', 'TX', 'MO']

print('Ready.')

## 1. Load Data

**Option A** – Load a CSV/Excel file you've already downloaded from AgendaWatch  
**Option B** – Scrape AgendaWatch live (requires MuckRock credentials in `config.py`)

In [ ]:
# ── Option A: local file ───────────────────────────────────────────────────────
DATA_FILE = '../data/agendawatch_co_tx_mo.csv'   # ← your file path here

az = ElectionAnalyzer.from_file(
    DATA_FILE,
    keyword=KEYWORD,
    states=STATES,
    filter_keyword=False,   # False = keep all docs; True = only docs mentioning keyword
)

# ── Option B: live scrape (uncomment to use) ───────────────────────────────────
# import os
# os.environ['AW_EMAIL']    = 'you@example.com'
# os.environ['AW_PASSWORD'] = 'yourpassword'
# az = ElectionAnalyzer.from_live(keyword=KEYWORD, states=STATES)

print(f'Loaded {len(az.df):,} documents')
az.df.head()

## 2. Build Corpus (tokenise + compute keyword stats)

In [ ]:
az.build_corpus()

c = az.corpus
print(f"Documents with '{KEYWORD}': {c.df['has_keyword'].sum():,}")
print(f"Total '{KEYWORD}' mentions: {c.df['kw_count'].sum():,}")
print()
print('Sample documents:')
az.sample_docs(3)

## 3. Train Word Embeddings (Word2Vec + TF-IDF)

In [ ]:
# This step may take 30–120 seconds depending on corpus size
az.build_model()
print('Model ready.')

## 4a. Semantic Neighbours

> **What words live in the same semantic space as 'election'?**

These are words that appear in *similar contexts* across meeting documents — not
just words that literally co-occur, but words whose surrounding language patterns
are similar. This reveals the conceptual landscape around election discussions.

In [ ]:
# Table of top neighbours
nbrs = az.neighbours(n=25)
display(nbrs)

# Try any word in the vocabulary:
# az.neighbours(word='fraud', n=20)

In [ ]:
# Bar chart
fig = az.neighbours_chart(n=20, save=True)
fig.show()

## 4b. 2-D Word Embedding Map

> Each dot = one word. **Spatial distance ≈ semantic similarity.**  
> Red dots are your seed words. Blue dots are the most similar words.

Clusters reveal which *topics* naturally group together in election discussions.

In [ ]:
# You can add extra seed words to highlight related concepts
SEED_WORDS = [KEYWORD, 'fraud', 'ballot', 'voter', 'integrity', 'certification']

fig = az.embedding_chart(
    n_words=250,
    seed_words=[w for w in SEED_WORDS if az.model.in_vocabulary(w)],
    save=True,
)
fig.show()

## 5. Keyword Trend Over Time

> **How has election-related language evolved across meeting documents?**

In [ ]:
# Monthly trend (change freq='Q' for quarterly, 'Y' for yearly)
fig = az.trend_chart(freq='M', metric='kw_per_doc', save=True)
fig.show()

In [ ]:
# View the underlying data
az.trend_data(freq='Q')

## 6. Top Jurisdictions

> **Which counties/cities/districts mention elections most?**

In [ ]:
# kw_per_1k = mentions per 1,000 words (normalised — better for comparison)
fig = az.jurisdiction_chart(metric='kw_per_1k', save=True)
fig.show()

In [ ]:
# Raw mention counts
fig2 = az.jurisdiction_chart(metric='kw_mentions', save=False)
fig2.show()

## 7. State Comparison

In [ ]:
fig = az.state_chart(save=True)
fig.show()

## 8. Co-occurrence Analysis

> **What words are most often used *near* 'election' in the documents?**

Unlike semantic similarity (which uses vector space), co-occurrence is purely
statistical — it counts how often other words appear within ±10 tokens of your
keyword. Good for surfacing direct context phrases.

In [ ]:
fig = az.cooccurrence_chart(n=30, save=True)
fig.show()

## 9. Most Keyword-Dense Documents (TF-IDF)

In [ ]:
az.top_docs(n=10)

## 10. Export

In [ ]:
# Export enriched corpus (with kw_count, kw_per_1k, tokens, etc.)
path = az.export_csv()
print(f'Exported to: {path}')

# Save Word2Vec model for later
# az.save_model()

## 11. Custom Keyword Exploration

Change `QUERY_WORD` to explore any term in the vocabulary.

In [ ]:
QUERY_WORD = 'fraud'   # ← try: 'ballot', 'voter', 'interference', 'noncitizen'

if az.model.in_vocabulary(QUERY_WORD):
    display(az.neighbours(n=20, word=QUERY_WORD))
else:
    print(f"'{QUERY_WORD}' not in vocabulary — try a different word or lower min_count in config.py")